# Lesson 8 · 🎨 Making rotation visible — the unit circle, conjugates and phase on the complex plane

A visual companion to L04–L07 (colors unified by `xviz`).

**Learning goals**
1. Point out, on a plot, rotation on the unit circle and where the roots of unity sit
2. Implement `plot_conjugate(z)`: draw z and its conjugate z̄ as mirror images across the real axis
3. Be able to explain out loud why the spectrum of a real signal is conjugate-symmetric (the second half mirrors the first)

**Why it matters for Aurora**: the DFT twiddle factors are simply points on the unit circle in the complex plane; once this picture makes sense, the twiddle factors in L37–L39 stop being abstract.

> **How to read this lesson**: the formula lessons (L04–L07) are behind you; this lesson is mostly **pictures**.
> **The genuinely new concept** is mainly the **conjugate** and its symmetry — most of the earlier sections are about "seeing what you've already learned". Feel free to skim the review parts.

← **Previous lesson** · [L07 · Everything is made of sines](L07_fourier_intuition.ipynb)

> Last lesson covered **everything is made of sines**: building a square wave out of a sum of sine waves, setting up the Fourier intuition in one picture.
> This lesson explores **visualizing the complex plane**.

## This lesson's story: making rotation visible

Math formulas are precise, but sometimes you need to **see** something before you truly understand it.

e^{iθ} = cosθ + i·sinθ is Euler's formula. But from reading the formula alone, many people only remember the symbols — without ever building the intuition that "a complex number = a point on a circle, multiplication = rotation".

This is a purely visual lesson: **the same concepts, told in pictures**.

**Suggested main path (follow this one if you're short on time)**:
1. Look at the plots of the unit circle / rotation / magnitude and phase (review of L04–L06, mostly pictures)
2. **The new core of this lesson**: the conjugate as a mirror image + `plot_conjugate`
3. Conjugate symmetry in one sentence + a code check (the rigorous proof can wait)

Here's what you'll see (items marked ★ are new in this lesson):
- Rotation on the unit circle → a sine wave is its "shadow" (recap of L04/L06)
- The three elements of a sinusoid → amplitude, frequency, phase (L04)
- Magnitude and phase of a complex number → the arrow's length and direction (L05)
- Roots of unity → the DFT twiddle factors on the circle (L06/L37)
- Adding up to a square wave → more terms, steeper edges (L07)
- ★ **The conjugate z̄**: a mirror image across the real axis; half of a real signal's spectrum mirrors out the other half

The one exercise in this lesson: implement `plot_conjugate(z)` — draw the symmetry between z and its conjugate z̄ on the complex plane.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
try:
    from xviz import (style, unit_circle_euler, sinusoid_anatomy,
                      complex_point, roots_of_unity, fourier_square)
    style()
    print('xviz ready')
except ImportError:
    import numpy as np
    import matplotlib.pyplot as plt
    import aurora.aviz as _aviz

    def unit_circle_euler(theta):
        """The point on the unit circle at angle theta, with its coordinates labeled."""
        fig, ax = plt.subplots(figsize=(5, 5))
        t_full = np.linspace(0, 2*np.pi, 300)
        ax.plot(np.cos(t_full), np.sin(t_full), plt.rcParams["grid.color"], lw=1, alpha=0.3)
        ax.axhline(0, color=plt.rcParams["axes.edgecolor"], lw=0.5); ax.axvline(0, color=plt.rcParams["axes.edgecolor"], lw=0.5)
        z = np.exp(1j * theta)
        ax.plot([0, z.real], [0, z.imag], '#2A9D8F', lw=2)
        ax.scatter([z.real], [z.imag], color='#E76F51', s=80, zorder=5)
        ax.annotate(f'  (cos θ, sin θ)\n= ({z.real:.3f}, {z.imag:.3f})',
                    xy=(z.real, z.imag), fontsize=10)
        ax.set_aspect('equal'); ax.set_title(f'e^{{iθ}}, θ = {theta:.3f} rad')
        plt.tight_layout(); plt.show()

    def sinusoid_anatomy():
        """The three elements of a sine wave: amplitude, frequency, phase."""
        t = np.linspace(0, 2, 500)
        fig, ax = plt.subplots(figsize=(10, 3))
        y = 1.5 * np.sin(2*np.pi*2*t + np.pi/4)
        ax.plot(t, y, '#2A9D8F', lw=2)
        ax.axhline(0, color=plt.rcParams["axes.edgecolor"], lw=0.5)
        ax.annotate('amplitude A=1.5', xy=(0.15, 1.5), fontsize=10, color='#E76F51')
        ax.annotate('period T=0.5s → f=2Hz', xy=(0.9, 0.2), fontsize=10, color=plt.rcParams["text.color"])
        ax.set_title('Three elements of a sine wave: A·sin(2πft + φ)'); plt.tight_layout(); plt.show()

    def complex_point(z):
        """A complex number z shown on the complex plane."""
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.axhline(0, color=plt.rcParams["axes.edgecolor"], lw=0.5); ax.axvline(0, color=plt.rcParams["axes.edgecolor"], lw=0.5)
        ax.plot([0, z.real], [0, z.imag], '#2A9D8F', lw=2)
        ax.scatter([z.real], [z.imag], color='#E76F51', s=80, zorder=5)
        ax.annotate(f'  {z.real:+.1f}{z.imag:+.1f}j\n  |z|={abs(z):.2f}, ∠={np.degrees(np.angle(z)):.1f}°',
                    xy=(z.real, z.imag), fontsize=10)
        lim = abs(z) * 1.4
        ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
        ax.set_aspect('equal'); ax.set_title(f'Complex plane: z = {z}')
        plt.tight_layout(); plt.show()

    def roots_of_unity(N):
        """N roots of unity = the DFT twiddle factors."""
        _aviz.twiddles(N)

    def fourier_square(ns):
        """Fourier-series approximation of a square wave."""
        t = np.linspace(0, 1, 1000)
        fig, ax = plt.subplots(figsize=(10, 4))
        for n in ns:
            y = sum(np.sin(2*np.pi*k*t)/k for k in range(1, n+1, 2)) * 4/np.pi
            ax.plot(t, y, label=f'{n} harmonics', alpha=0.7)
        ax.legend(); ax.set_title('Fourier-series approximation of a square wave')
        plt.tight_layout(); plt.show()

    print('⚠️  xviz not installed; switched to the aurora.aviz fallback (same functionality)')

## 0. Groundwork: what is i? And what are radians?

Before we really get going, let's settle two "foundation-level" questions — they aren't the focus of this lesson, but if the foundation is shaky, everything after it looks like a magic trick.

**① Why invent i at all?**

You learned in middle school: no real number squared is ever negative (a positive squared is positive, a negative squared is also positive, and 0 squared is 0). Yet when mathematicians solve equations, they keep running into "unsolvable" ones like x² + 1 = 0 — and by the old rules, it really has no solution.

The mathematicians' move was a bit of a "cheat", but a very effective one: since there's no real number whose square is −1, just **define** a new symbol i, declare that i² = −1, and see what happens. The result: this symbol "conjured out of thin air" turned rotations in the plane, alternating current, and quantum mechanics into clean formulas. It isn't word play — it's one of the few cases in the history of mathematics of "build the tool first, discover later that it's incredibly useful".

With i in hand, any complex number can be written as a + bi: a is the "real part" (the horizontal coordinate), and b is the "imaginary part" (the vertical coordinate). Together, a + bi is just a point (a, b) in the plane — which is also the starting point of this lesson's main thread, "complex number = point in the plane".

**② What are radians, and why does Python use `np.pi/2` for 90°?**

Degrees ("360° in a full turn") are a human convention: split a circle into 360 parts, purely because the Babylonians liked the number 360 (lots of divisors, divides evenly).

Radians measure angles in a more "mathematically native" way: **arc length divided by radius**. Walk once around the unit circle (radius 1) and the arc length is exactly the circumference 2πr = 2π. So "one full turn" = 2π radians = 360°; "a quarter turn" (90°) = 2π/4 = π/2 radians. That's why `np.pi/2` stands for 90° in code.

You only need to remember one conversion: **radians = degrees × π/180** (and back: degrees = radians × 180/π). The code below lines up a few common angles in both forms — run it once and it'll click.

In [ ]:
import numpy as np

# The imaginary unit: by definition, i**2 = -1
i = 1j
print('i =', i, '  i**2 =', i**2)

# Degrees <-> radians lookup table
degrees_list = [0, 30, 45, 90, 180, 270, 360]
for d in degrees_list:
    r = np.radians(d)          # degrees -> radians
    back = np.degrees(r)       # radians -> degrees (check that the conversion round-trips)
    print(f'{d:>4}°  =  {r:.4f} rad   (back to degrees: {back:.1f}°)')

## Into the lab: angles, coordinates and rotation

`z = np.exp(1j * angles)` returns the real part (cos) and imaginary part (sin) for a whole array of angles in one go. When you run it, notice how `z.real` and `z.imag` take turns rising and falling as the angle changes.

## Hands-on: a complex number is "a position in a rotation"

The key to complex numbers and trig isn't the pile of symbols — it's that they write a rotation in the 2-D plane as a single number. Let's first look at the complex coordinates for a few angles.

In [ ]:
import numpy as np

angles = np.array([0, np.pi/2, np.pi, 3*np.pi/2])
z = np.exp(1j * angles)

print('angle =', np.round(angles, 3))
print('real part cos =', np.round(z.real, 3))
print('imag part sin =', np.round(z.imag, 3))
print('complex z =', np.round(z, 3))


## Code experiment: one full turn

Split `2π` into 8 equal steps and check that the coordinates of `exp(1j*theta)` land on the unit circle following the cos/sin pattern.

In [ ]:
import numpy as np

for k in range(9):
    theta = 2 * np.pi * k / 8
    z = np.exp(1j * theta)
    print(f'k={k} theta={theta:.2f} -> ({z.real:+.3f}, {z.imag:+.3f})')


## 1. Euler's formula: e^{iθ} is rotation on the unit circle (see L06)

The real part of `e^{iθ}` is `cos θ` and the imaginary part is `sin θ` — as the angle θ sweeps from 0 to 2π, the point travels once all the way around the unit circle. Multiplying complex numbers is equivalent to rotating in the plane: when two complex numbers multiply, their magnitudes (r) multiply and their angles add — that's the geometric meaning of Euler's formula.

This visual companion puts the core plots from four lessons — the three elements of a sinusoid (L04), magnitude and phase of complex numbers (L05), Euler's formula (L06), and Fourier sums (L07) — into the same `xviz` color scheme, so the rise and fall of a sine wave and the projection of a rotating vector sit on the same picture, and you don't have to flip back and forth between four files to see how they connect.

### Why does e^{iθ} equal exactly cosθ + i·sinθ? (review / skippable, covered in L06)

> Main path: if L06 already set up the "definition contract" for you, you can skim this whole section and go straight to checking the plots.

This is the step in the lesson most likely to leave people "dazed": e is the base of the natural logarithm (about 2.71828), a perfectly ordinary real number; θ is an angle. What on earth does "a real number raised to an imaginary power" mean? And why does the answer come out exactly cosθ + i·sinθ, no more, no less?

**The key to the answer: e^x is itself a sum of infinitely many terms (its Taylor series)** — we're just so used to the symbol e^x that we forget this is how it's actually defined underneath:

e^x = 1 + x + x²/2! + x³/3! + x⁴/4! + x⁵/5! + ...

(This is a result from higher math; for now, accept it and use it. Think of it as "approximating the curve e^x layer by layer with infinitely many polynomial terms", like building a round mosaic out of more and more little tiles — the more tiles, the closer to a circle.)

Now replace x with iθ and watch the magic happen:

e^{iθ} = 1 + iθ + (iθ)²/2! + (iθ)³/3! + (iθ)⁴/4! + ...

Because i² = −1, the powers of i cycle every 4 steps: i, −1, −i, 1 (i¹=i, i²=−1, i³=−i, i⁴=1, i⁵=i, ...). Sort the terms into two piles, "real terms" and "imaginary terms with an i":

- Terms without i: 1 − θ²/2! + θ⁴/4! − θ⁶/6! + ...  → that's exactly the Taylor series of cosθ!
- Terms with i: i·(θ − θ³/3! + θ⁵/5! − θ⁷/7! + ...)  → what's in the parentheses is exactly the Taylor series of sinθ!

Add the two piles together and you get e^{iθ} = cosθ + i·sinθ — not a coincidence, not a convention, but the infinite series splitting apart and lining up perfectly. The formula was discovered by the 18th-century mathematician Euler, hence the name Euler's formula.

You don't need to memorize this derivation, but **knowing "there's a rigorous proof behind it, it wasn't just made up" is the key step from "rote-memorizing a formula" to "feeling sure of it"**. The code below checks it: add up only a finite number of terms of the series, and the result really does get closer and closer to cosθ + i·sinθ.

In [ ]:
import math
import numpy as np

theta = np.pi / 3  # 60°, just some angle to test with

def euler_by_series(theta, n_terms):
    """Approximate e^{i*theta} with the first n_terms terms of its Taylor series."""
    total = 0
    for n in range(n_terms):
        total += (1j * theta) ** n / math.factorial(n)
    return total

for n_terms in [1, 2, 3, 5, 8, 15]:
    approx = euler_by_series(theta, n_terms)
    exact = np.exp(1j * theta)
    print(f'only {n_terms:>2} terms: {approx:.6f}   error = {abs(approx - exact):.2e}')

print()
print(f'exact e^(iθ)  = {np.exp(1j*theta):.6f}')
print(f'cosθ + i·sinθ = {np.cos(theta) + 1j*np.sin(theta):.6f}  (the two should match exactly)')

### Why does multiplying complex numbers "multiply the magnitudes and add the angles"? — magnitude and angle are two separate things

Above we said "when two complex numbers multiply, their magnitudes multiply and their angles add". The "magnitude" here (modulus, i.e. the length of the arrow |z|) and the "angle" (the direction the arrow points) are two independent properties of a complex number, and multiplication acts on each of them separately:

- **Angle only**: if both numbers have magnitude 1 (both on the unit circle), multiplication is pure rotation. For example, z₁ = 1 (angle 0°), z₂ = i (angle 90°, since i is exactly the point on the unit circle turned by 90°). z₁ × z₂ = i: the effect is "turn 1 by 90°", and the result is still on the unit circle (magnitude still 1).
- **What if the magnitude isn't 1**: if z₂ = 2i (magnitude 2, angle still 90°), then z₁ × z₂ = 2i: the effect is "turn 1 by 90°, then stretch it by a factor of 2". Now multiplication is doing two things — changing direction + changing length.

So, strictly speaking, "complex multiplication = rotation" should be "complex multiplication = rotation + scaling": the angle part handles the direction, the magnitude part handles the size; only when the magnitudes are all 1 (all on the unit circle) is multiplication "pure" rotation with no scaling involved. The twiddle factors the DFT uses (roots of unity) all lie on the unit circle, with magnitude always 1 — that's why Section 4 calls them "pure rotations": no scaling muddying the picture.

In [ ]:
import numpy as np

z1 = 1 + 0j
for z2, label in [(1j, 'z2 = i  (|z|=1, angle=90°)'), (2j, 'z2 = 2i (|z|=2, angle=90°)')]:
    product = z1 * z2
    print(label)
    print(f'  z1 × z2 = {product}')
    print(f'  magnitude: |z1|={abs(z1):.1f} × |z2|={abs(z2):.1f} = {abs(z1)*abs(z2):.1f}'
          f'  (should equal |z1*z2|={abs(product):.1f})')
    print(f'  angle: {np.degrees(np.angle(z1)):.0f}° + {np.degrees(np.angle(z2)):.0f}°'
          f' = {np.degrees(np.angle(z1))+np.degrees(np.angle(z2)):.0f}°'
          f'  (should equal the angle of z1*z2, {np.degrees(np.angle(product)):.0f}°)')
    print()

In [ ]:
import numpy as np
unit_circle_euler(np.pi/3);

## 2. The three elements of a sinusoid: amplitude / frequency / phase (see L04)

In [ ]:
sinusoid_anatomy();

## 3. A complex number = a point in the plane: magnitude + phase (see L05)

**🔗 Aurora**: the FFT output of `src/aurora/audio/transforms.py` is an array of complex numbers; take `np.abs()` of it to get the magnitude spectrum (how strong the energy is at each frequency), and `np.angle()` to get the phase spectrum (how the signal is aligned in time at each frequency). Every value of `twiddle(k, n, N)` in `notebooks/1_complex_trig/L06_euler.ipynb` is a twiddle factor sitting on the unit circle — fix the frequency index k and let n sweep over 0..N-1 to get a column of twiddle factors; its inner product with the input signal is the complex result for the corresponding frequency bin.

### Too many symbols? First, get to know who i, k, n and N each are

The paragraph above suddenly threw out expressions like `twiddle(k, n, N)` and `e^{−2πikn/N}` — four letters piled together are easy to blur. Here's an ID card for each letter. You don't need to understand the whole formula yet; you only need to know "who each letter is":

| Symbol | What it is | Analogy |
|---|---|---|
| **i** | the imaginary unit, i² = −1 | the same i from the "groundwork" section earlier |
| **N** | how many samples the signal has in total | how many "snapshots" a recording took in total |
| **n** | the time index, taking values 0, 1, ..., N−1 | which "snapshot" (which step in time) |
| **k** | the frequency index, taking values 0, 1, ..., N−1 | which "rhythm" (which frequency) |

What `e^{−2πikn/N}` is saying is: "for the k-th frequency and the n-th time point, where on the unit circle has the twiddle factor turned to?" — k and n together determine the angle, and N determines how many slices "one full turn" is cut into.

This formula and the word `twiddle` are the stars of L37–L39, where we write the FFT by hand. In this lesson you only need them to look familiar — know what the formula looks like and who each letter is. You don't need to derive or compute it yet — **the detailed math is saved for L37–L39, where it'll be explained again step by step**. It's making an early appearance here so it won't feel strange when you meet it then.

In [ ]:
complex_point(3+4j);

## 3b. The complex conjugate: a reflection across the real axis

The conjugate of a complex number `z = a + bj` is `z̄ = a − bj`: flip the sign of the imaginary part. Geometrically, that's the same as reflecting the point across the real axis.

**Checking the properties**:
- `z + z̄ = 2a` (twice the real part; the imaginary parts cancel)
- `z × z̄ = a² + b² = |z|²` (the squared magnitude, a pure real number)

**Aurora connection**: the DFT of a real-valued signal satisfies conjugate symmetry `X[N−k] = conj(X[k])`, so you only need to store the first half of the spectrum. This is the basis for compressing the FFT output in `src/aurora/audio/transforms.py`.

### Why is it called a "conjugate"? What's the point of the mirror reflection?

The word "conjugate" sounds abstract, but it comes from the Latin *conjugare*, "to yoke together" — like two oxen joined under one yoke so they always work as a pair. z and z̄ come in a pair the same way: as soon as z = a + bi is given, z̄ = a − bi is uniquely determined along with it. The two always appear together, as mirror images of each other.

On a plot, concretely: z = 3 + 4i corresponds to the point (3, 4) in the plane; z̄ = 3 − 4i corresponds to the point (3, −4). The two points have exactly the same horizontal coordinate (real part), and opposite vertical coordinates (imaginary parts) — on paper, that's flipping (3,4) down across the horizontal (real) axis so it lands exactly on (3,−4), just like looking in a mirror (with the mirror standing on the real axis).

**Why this operation matters (a spoiler — L38 covers it in detail)**: a real sound signal (say, the air-pressure changes picked up by a microphone) is always real-valued, with no imaginary part. But after you take its DFT, the result is complex. Remarkably, the "left half" and the "right half" of this complex spectrum are conjugates of each other — in other words, once you know the first half of the spectrum, you can "mirror" out the second half directly with the conjugate, no need to compute it again. That's why, when processing real recordings, a program only needs to store the first half of the spectrum, saving half the storage and computation. The next few cells fill in where this property comes from.

### Why does z × z̄ equal exactly |z|²? — expand the product honestly

The lesson's claim is `z × z̄ = a² + b² = |z|²`. The step looks simple, but to be convinced of "why", you need to expand it yourself — the formula didn't fall from the sky:

Let z = a + bi and z̄ = a − bi. Expand (a+bi)(a−bi) just like the polynomial multiplication (distributive law) you learned in school:

(a + bi)(a − bi)
= a·a + a·(−bi) + bi·a + bi·(−bi)
= a² − abi + abi − b²i²

The −abi and +abi in the middle cancel exactly (that's the neat trick of multiplying conjugates: the cross terms always cancel), which leaves:

= a² − b²i²

Now substitute i² = −1:

= a² − b²×(−1) = a² + b²

And a² + b² is exactly |z| squared (Pythagorean theorem: the distance from z to the origin is |z| = √(a²+b²)). So z × z̄ = a² + b² = |z|² — not a convention, but a result that falls out naturally once you expand. The code below lays out these steps line by line to check them.

In [ ]:
a, b = 3, 4
z = a + b*1j
z_conj = a - b*1j

# Expand (a+bi)(a-bi) term by term
term1 = a * a             # a·a
term2 = a * (-b*1j)       # a·(-bi)
term3 = (b*1j) * a        # bi·a
term4 = (b*1j) * (-b*1j)  # bi·(-bi)

print(f'a·a        = {term1}')
print(f'a·(-bi)    = {term2}')
print(f'bi·a       = {term3}   ← the negative of the line above; they cancel when added')
print(f'bi·(-bi)   = {term4}   = -b²i² = -b²×(-1) = b²')
print()
print(f'sum of four terms = {term1 + term2 + term3 + term4}')
print(f'z × z̄ (direct)    = {z * z_conj}')
print(f'a² + b²           = {a**2 + b**2}')
print(f'|z|²              = {abs(z)**2}')

### Why does the conjugate symmetry X[N−k] = conj(X[k]) hold?

> **Main text — just remember the conclusion**: a recording is real-valued → the second half of its spectrum is the conjugate mirror of the first half → you usually only need to store half.
> Below is a simplified proof (it appears again formally in L38); if you're in a hurry, run the code check first and dig into the proof "when you have time".

This property looks like a conclusion dropped out of nowhere, but it actually follows from the same trick behind z×z̄=|z|², which we just proved (i²=−1 and the definition of the conjugate). Here's a simplified line of thought (the formal notation returns in L38; for now we're building intuition):

The k-th frequency bin of the DFT is defined as a weighted sum over all samples n: X[k] = Σₙ x[n] · e^{−i2πkn/N} (n from 0 to N−1).

Now look at bin N−k, replacing k with N−k:

X[N−k] = Σₙ x[n] · e^{−i2π(N−k)n/N} = Σₙ x[n] · e^{−i2πn} · e^{i2πkn/N}

Notice that in e^{−i2πn}, n is an integer, so −i2πn means going around the unit circle a whole number of times and ending up back where you started — so e^{−i2πn} = 1 (like a clock hand going around several full times and pointing back at the same spot). So:

X[N−k] = Σₙ x[n] · e^{i2πkn/N}

Now take the conjugate of X[k] = Σₙ x[n] · e^{−i2πkn/N} and compare: since x[n] is **real** (no imaginary part, so the conjugate leaves it unchanged), conjugating just flips the sign of i in the exponent:

conj(X[k]) = Σₙ x[n] · e^{i2πkn/N}

The right-hand sides of the two equations are identical, so X[N−k] = conj(X[k]) — **the key requirement is that x[n] is a real signal**. That's also why this property holds only for real recordings (real-valued signals), not for complex signals in general. The code below checks it on a random "recording".

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
N = 16
x = rng.standard_normal(N)       # simulate a real (purely real-valued) recording
X = np.fft.fft(x)                # compute the spectrum with numpy's built-in FFT (used directly for now; we write it by hand in L37-L39)

for k in [1, 3, 7]:
    print(f'X[{k}]        = {X[k]:.4f}')
    print(f'conj(X[{k}])  = {np.conj(X[k]):.4f}')
    print(f'X[{N-k}]       = {X[N-k]:.4f}   (should equal conj(X[{k}]))')
    print()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_conjugate_demo(z):
    """Demo: the symmetry between a complex number z and its conjugate z̄ on the complex plane."""
    z_conj = z.conjugate()
    fig, ax = plt.subplots(figsize=(5, 5))
    t = np.linspace(0, 2*np.pi, 300)
    ax.plot(np.cos(t)*abs(z), np.sin(t)*abs(z), plt.rcParams["grid.color"], lw=0.5, alpha=0.3, ls='--')
    ax.axhline(0, color=plt.rcParams["axes.edgecolor"], lw=0.8)
    ax.axvline(0, color=plt.rcParams["axes.edgecolor"], lw=0.5)
    # z
    ax.plot([0, z.real], [0, z.imag], '#2A9D8F', lw=2)
    ax.scatter([z.real], [z.imag], color='#2A9D8F', s=80, zorder=5,
               label=f'z = {z.real:+.2f}{z.imag:+.2f}j')
    # z̄
    ax.plot([0, z_conj.real], [0, z_conj.imag], '#E76F51', lw=2, ls='--')
    ax.scatter([z_conj.real], [z_conj.imag], color='#E76F51', s=80, zorder=5,
               label=fr'$\bar{{z}}$ = {z_conj.real:+.2f}{z_conj.imag:+.2f}j')
    ax.set_aspect('equal')
    ax.legend(loc='lower right', fontsize=9)
    ax.set_title(fr'Conjugate symmetry  $|z| = |\bar{{z}}| = {abs(z):.2f}$   $z\times\bar{{z}} = {(z*z_conj):.2f}$')
    plt.tight_layout(); plt.show()
plot_conjugate_demo(3 + 4j)

## 🏋️ Exercise: implement plot_conjugate

**Task**: complete the function below so it draws `z` and `z̄` on the same axes and shows their magnitude in the plot title.

**Three checkpoints**:
1. `z.conjugate().imag == −z.imag` (imaginary part flipped)
2. `abs(z) == abs(z.conjugate())` (magnitude unchanged)
3. `(z * z.conjugate()).imag ≈ 0` (the product is purely real)

### How to write it: annotate, quiver or arrow? How to set the axis limits?

The TODO hint says "use `ax.annotate` or `ax.quiver` to draw arrows", but the reference implementation in `plot_conjugate_demo` above actually uses `ax.plot()` (a line segment) + `ax.scatter()` (a dot) — all three approaches give the visual effect of an "arrow"; any of them is fine, they just work differently:

- **`ax.plot([0, z.real], [0, z.imag], ...)` + `ax.scatter(...)`**: draw a line segment from the origin to z, then put a dot at the end. The most straightforward; this is what `plot_conjugate_demo` uses.
- **`ax.annotate('', xy=(z.real, z.imag), xytext=(0,0), arrowprops=dict(arrowstyle='->'))`**: `annotate` is really meant for adding text labels, but leave the text empty and add `arrowprops`, and it becomes an arrow-drawing tool. The arrow gets an arrowhead, so it looks more like a "vector" than `plot`'s straight line.
- **`ax.quiver(0, 0, z.real, z.imag, angles='xy', scale_units='xy', scale=1, ...)`**: `quiver` is a function made for drawing vector fields; it can draw many arrows at once and takes a few more parameters (`angles='xy', scale_units='xy', scale=1` is a fixed combination that makes the arrow length match the axis units — just remember to copy these).

All three look similar; use whichever feels comfortable. Below is a small example unrelated to z and z̄, just to show what the parameters of these three functions look like (it is not the answer to the exercise):

**Axis limits**: you can borrow the approach from the fallback `complex_point` in this lesson's opening import cell (the `except ImportError` branch) — `lim = abs(z) * 1.4`, then `ax.set_xlim(-lim, lim)` and `ax.set_ylim(-lim, lim)`. This scales automatically with |z|, so the arrow always fits completely no matter how long it is, with a bit of margin on each side (1.4× is a rule of thumb that leaves enough room for the arrow's label text).

**Colors**: `#2A9D8F` (teal) and `#E76F51` (orange-red) are the two high-contrast colors used throughout this lesson (the xviz color scheme), standing for z and z̄ respectively. Just keep them consistent with `plot_conjugate_demo` above; no need to design your own palette.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

v = (2, 1)  # just an example vector for the demo, unrelated to the z in the exercise

axes[0].plot([0, v[0]], [0, v[1]], color='#2A9D8F', lw=2)
axes[0].scatter([v[0]], [v[1]], color='#2A9D8F', s=80)
axes[0].set_title('ax.plot + ax.scatter')

axes[1].annotate('', xy=v, xytext=(0, 0),
                  arrowprops=dict(arrowstyle='->', color='#E76F51', lw=2))
axes[1].set_title('ax.annotate(arrowprops=...)')

axes[2].quiver(0, 0, v[0], v[1], angles='xy', scale_units='xy', scale=1,
               color=plt.rcParams["text.color"], width=0.02)
axes[2].set_title('ax.quiver')

for ax in axes:
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
    ax.axhline(0, color=plt.rcParams["axes.edgecolor"], lw=0.5); ax.axvline(0, color=plt.rcParams["axes.edgecolor"], lw=0.5)
    ax.set_aspect('equal')

plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_conjugate(z):
    """Draw z and z̄ on the same axes; the plot title shows their magnitude."""
    # ✏️ TODO:
    #   1. compute the conjugate: z_conj = z.real - 1j * z.imag
    #   2. compute the magnitude: mag = abs(z)
    #   3. fig, ax = plt.subplots()
    #   4. draw arrows for z and z_conj with ax.annotate or ax.quiver
    #   5. ax.set_title(f'|z| = {mag:.2f}')
    raise NotImplementedError("TODO: implement plot_conjugate")

# ── Once the function is done, uncomment the lines below ──
# plot_conjugate(3 + 4j)
# plot_conjugate(1 + 1j)
# plot_conjugate(-2 + 3j)

# ── Automatic asserts (run once the function is implemented to verify) ──
z_test = 3 + 4j
z_bar  = z_test.conjugate()
try:
    plot_conjugate(z_test)
    assert z_bar.imag == -z_test.imag,       f"imaginary part not flipped: {z_bar.imag} != {-z_test.imag}"
    assert abs(z_test) == abs(z_bar),         f"magnitudes should be equal: {abs(z_test)} != {abs(z_bar)}"
    assert abs((z_test * z_bar).imag) < 1e-10, f"z×z̄ should be purely real, imaginary part = {(z_test * z_bar).imag}"
    print("✅ All asserts passed")
except (NotImplementedError, TypeError):
    print("⏳ Implement plot_conjugate(z) first, then run the asserts.")

In [ ]:
# Mathematical properties of the conjugate (checking formulas; this is not a whiteboard answer cell)
# Check the math behind plot_conjugate (no need to actually call the plotting function)
import numpy as np

# Properties of the conjugate: same real part, opposite imaginary part, equal magnitude
z = 3 + 4j
z_conj = z.real - 1j * z.imag
assert z_conj == (3 - 4j),            f"conjugate should be 3-4j, got {z_conj}"
assert abs(z_conj) == abs(z),          f"magnitudes should be equal: |z|={abs(z)}, |z̄|={abs(z_conj)}"
assert np.isclose(z * z_conj, abs(z)**2), f"z·z̄ = |z|² should be {abs(z)**2}"
print(f"Conjugate check ✅")
print(f"  z     = {z}  |z| = {abs(z)}")
print(f"  z̄     = {z_conj}  |z̄| = {abs(z_conj)}")
print(f"  z·z̄   = {z * z_conj:.1f}  = |z|² = {abs(z)**2:.1f}")
print("  Conclusion: the conjugate keeps the magnitude and only flips the sign of the imaginary part (mirror symmetry across the real axis)")

## 4. Roots of unity = the DFT twiddle factors (see L06 / the FFT in L37-L39)

### Why "roots of unity"? How do you read W_N^k?

In math, the word "root" usually means "a solution of an equation" (for example, "the roots of a quadratic" are the x values that make it equal 0). A "root of unity" means the same thing: **it's a solution of the equation zᴺ = 1** — "unity" refers to the 1 on the right-hand side (also called the "identity element"), and the "roots" are the z values you solve for.

For example, with N=4, solving z⁴=1 gives 4 solutions: 1, i, −1, −i (you can check that i⁴ = i²×i² = (−1)×(−1) = 1). These 4 solutions are spread evenly around the unit circle, 90° apart. For any N, the equation zᴺ=1 has exactly N solutions, spread evenly around the unit circle 360°/N apart — those are the points you see in this lesson's plots.

**How to read the symbol `W_N^k`**: the subscript N means "this is the set of roots of unity that cuts the unit circle into N parts", and the superscript k means "take the k-th one" (i.e. turned k steps). So `W_N^k = e^{−i2πk/N}`, and together the two mean "the k-th root of unity out of N". This notation is standard in signal-processing textbooks, and you'll use it every day when writing the FFT by hand in L37–L39.

The code below checks it: pick a few values of k, and raising `W_N^k` to the N-th power really does bring each one back to 1.

In [ ]:
import numpy as np

N = 8
for k in [1, 3, 5]:
    W = np.exp(-1j * 2 * np.pi * k / N)   # root of unity W_N^k
    W_to_N = W ** N
    print(f'k={k}:  W_{N}^{k} = {W:.4f}   (W_{N}^{k})^{N} = {W_to_N:.4f}  ≈ 1 ✓')

### Where does the minus sign come from? Why do the points have to be evenly spaced?

**Where the minus sign comes from**: `e^{iθ}` (plus sign) rotates counterclockwise, `e^{−iθ}` (minus sign) rotates clockwise — this is just a sign convention for angles, like choosing to walk along the positive or negative x-axis; there's no deeper physical meaning. The DFT definition uses a minus sign (`e^{−i2πkn/N}`) by convention when "analyzing" a signal; going the other way, the inverse transform (IDFT) that "synthesizes" a time-domain signal back from its spectrum uses a plus sign (`e^{+i2πkn/N}`). You can remember it as a handy mnemonic: **take it apart (analysis): minus, clockwise; put it back (synthesis): plus, counterclockwise** — the detailed derivation comes in L37–L39 when we write the FFT/IDFT by hand. For now, just know that "the minus sign isn't an arbitrary choice; it's the convention for the two directions, analysis and synthesis".

**Why evenly spaced — why can't we pick any N points?** — what does "rotation basis" mean?

The word "basis" will be properly introduced in L09 (the next lesson, linear algebra). For now, let's build intuition with an analogy: if you want to measure an object's position, and you measure only along two perpendicular directions, "due east" and "due north" (like the x-axis and y-axis from school), the two numbers you get pin down the point uniquely, and the two directions "don't interfere" (the eastward measurement doesn't affect the northward one). That's the intuition behind "orthogonal": the less the directions overlap and the more evenly they're chosen, the less redundant the measured information.

Roots of unity spread evenly around the circle are exactly such a set of "non-interfering" directions in the complex world: neighboring directions among the N are all separated by equal angles, the DFT uses these N directions to "measure" the signal, and the N frequency components it measures don't overlap — no information is wasted. If the N points were spaced unevenly (say, crowded on one side of the circle), some "directions" would be measured twice and others not at all, and that part of the frequency information would be lost — which is also why the DFT must use evenly spaced roots of unity rather than an arbitrary choice.

In [ ]:
roots_of_unity(8);

## 5. Adding sines → a square wave: Fourier series intuition (see L07)

**🔗 Aurora**: `fourier_square((1,3,9,50))` approximates a square wave by adding odd harmonics — exactly a truncated Fourier series. `twiddle(k, n, N)` in `notebooks/1_complex_trig/L06_euler.ipynb` extends the same decomposition to any discrete signal: what the FFT does is efficiently compute the projection coefficient of the signal onto each frequency component, and the inverse transform adds those components back up to recover the signal.

### What exactly are "projection coefficients" and the complex dot product?

Above we said "what the FFT does is efficiently compute the projection coefficient of the signal onto each frequency component". The "projection" here is actually the same concept as the vector dot product from high school, just upgraded from real vectors to complex vectors.

**First, recall the dot product of real vectors**: for two vectors a=(a₁,a₂) and b=(b₁,b₂), the dot product is a·b = a₁b₁ + a₂b₂. Geometrically, it's "the length of a projected onto the direction of b, times the length of b" — the larger the dot product, the closer the two vectors point in the same direction.

**The complex dot product has one extra step: take the conjugate of the second vector**. For two complex vectors z₁=(z₁,₀,...,z₁,ₙ) and z₂=(z₂,₀,...,z₂,ₙ), the dot product is defined as:

z₁ · z₂ = Σₘ z₁,ₘ × conj(z₂,ₘ)

**Why the conjugate?** — without it, the "length" you'd get from dotting a vector with itself could be complex, and you couldn't compare sizes (for example, i×i = −1, and a negative "length" makes no sense). With the conjugate, z×conj(z) = |z|² (the property we just proved!), which is always a non-negative real number, so the "length" from the dot product means something: you can compare sizes and judge "how close".

**Back to the FFT**: the projection coefficient of a signal on the k-th frequency component is the dot product of the signal x[n] (seen as a vector of length N) with the vector formed by the k-th root of unity, e^{i2πkn/N} (n from 0 to N-1). By the definition above, the root of unity gets conjugated, so the actual formula is Σₙ x[n]×conj(e^{i2πkn/N}) = Σₙ x[n]×e^{−i2πkn/N} — and that's another source of the minus sign in the DFT formula `e^{−i2πkn/N}` (the same thing as the "clockwise/counterclockwise" explanation in the previous section, seen from two angles).

In [ ]:
import numpy as np

# Toy example: two complex vectors of length 3, showing that "the dot product needs a conjugate"
z1 = np.array([1+1j, 2+0j, 0+1j])
z2 = np.array([1+0j, 1+1j, 1-1j])

dot_with_conj = np.sum(z1 * np.conj(z2))     # the correct way: conjugate z2
dot_without_conj = np.sum(z1 * z2)            # for comparison: no conjugate

print('z1 · conj(z2) =', dot_with_conj)
print('z1 · z2 (no conjugate, for comparison only) =', dot_without_conj)

# Check that dotting a vector with its own conjugate gives a non-negative real number
self_dot = np.sum(z1 * np.conj(z1))
print('z1 · conj(z1) =', self_dot, ' (should be a non-negative real number: the sum of the squared magnitudes of the components)')

In [ ]:
fourier_square((1,3,9,50));

In [ ]:
for k in range(8):
    theta = 2*np.pi*k/8
    z = np.exp(1j*theta)
    radius = abs(z)
    print(f'k={k} | z=({z.real:+.2f}, {z.imag:+.2f}) | radius={radius:.2f}')
print('Every point has radius close to 1, so they all lie on the unit circle.')


## Parameter experiment: change the number of roots of unity

In Section 4 (the roots-of-unity plot), change the argument `N` of `roots_of_unity(8)` from `8` to `16`. Watch the twiddle factors go from 8 to 16, the density of points spread evenly around the unit circle double, and the angle between neighboring points shrink from `2π/8 = 45°` to `2π/16 = 22.5°`. Whatever N is, `abs(z)` of every root of unity `W_N^k = e^{-i2πk/N}` is always 1, which guarantees that the column vectors of the DFT matrix form a rotation basis of equal magnitude.

## Wrap-up

You can now call `unit_circle_euler(theta)` to visualize any rotation angle on the unit circle, and use `roots_of_unity(N)` to generate the N twiddle factors `W_N^k = e^{-i2πk/N}` the DFT needs. This set of equally spaced complex points always has radius 1, and it is the geometric origin of the FFT butterfly operation. The twiddle-factor geometry of L06 (`twiddle`) matches the roots of unity in this lesson; L37–L39 will write the FFT / DFT matrix by hand. The complex value at each frequency bin of the DFT/FFT output in L37–L41 (L40 plots it as a spectrum) is the dot product of the signal with the corresponding root of unity; a real spectrogram only appears in L43–L45, by stacking these frame by frame.

The next lesson moves into linear algebra (L09), starting from vector spaces and matrix multiplication, to lay the groundwork for the matrix form of the DFT.

In [ ]:
# ✏️ Self-review for this lesson
l08_review = {
    "plot_conjugate_done":    None,  # plot_conjugate implemented and the asserts pass? True/False
    "unit_circle_intuition":  None,  # can you connect e^{iθ} with rotation on the unit circle? True/False
    "twiddle_geometry":       None,  # can you explain how roots of unity relate to the L06 twiddle? True/False
    "conjugate_properties_verified": None,  # the three conjugate properties (imaginary part flipped / magnitude unchanged / z·z̄ purely real) checked? True/False
}

unfilled = [k for k, v in l08_review.items() if v is None]
assert not unfilled, f'Not filled in yet: {unfilled}'
weak = [k for k, v in l08_review.items() if v is False]
if weak:
    print(f'⚠️  Needs more work: {weak}')
else:
    print('✅ L08 all cleared! On to L09: vector basics')


---

→ **Next lesson** · [L09 · Vector algebra](../2_linear_algebra/L09_vectors.ipynb)

> Next lesson covers **vector algebra**: addition, scalar multiplication and linear combinations — a NumPy implementation + the geometric meaning.